In [ ]:
# !pip install pyreadstat

In [ ]:
import pyreadstat
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score
from sklearn.metrics import precision_recall_curve, average_precision_score, PrecisionRecallDisplay
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from collections import Counter
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt


# EDA

In [ ]:
df_ch = pyreadstat.read_sav('/kaggle/input/malnutrition/ch.sav')[0]
# data = pd.read_spss('/kaggle/input/malnutrition/ch.sav')

# Renaming the columns
df1 = df_ch[['AN4', 'AN8', 'BD2', 'CA1', 'CA14', 'HH6', 'HL4', 'melevel1', 'cinsurance', 'HH7c', 'windex5', 'WAZ2', 'HAZ2', 'WHZ2']]
df1 = df1.rename(columns={
    'AN4': 'child_age',
    'AN8': 'child_weight',
    'BD2': 'ever_breastfed',
    'CA1': 'diarrhoea_last_2_weeks',
    'CA14': 'fever_last_2_weeks',
    'HH6': 'area',
    'HL4': 'child_sex',
    'melevel1': 'mother_education',
    'cinsurance': 'health_insurance',
    'HH7c': 'province',
    'windex5': 'wealth_index',
    'WAZ2': 'underweight',
    'HAZ2': 'stunting',
    'WHZ2': 'wasting'
})

coded_child_age = [99.6, 99.5, 99.4, 99.3]
coded_underweight = [99.99, 99.98, 99.97]
coded_stunting = [99.99, 99.98, 99.97]
coded_wasting = [99.97]
coded_diarrhoea = [9, 8]
coded_fever = [8]
coded_insurance = [9]
coded_breastfed = [9]

# Remove rows with coded entries in each column
df2 = df1.copy()
df2 = df2[~df2['child_age'].isin(coded_child_age)]
df2 = df2[~df2['underweight'].isin(coded_underweight)]
df2 = df2[~df2['stunting'].isin(coded_stunting)]
df2 = df2[~df2['wasting'].isin(coded_wasting)] 
df2 = df2[~df2['diarrhoea_last_2_weeks'].isin(coded_diarrhoea)]
df2 = df2[~df2['fever_last_2_weeks'].isin(coded_fever)]
df2 = df2[~df2['health_insurance'].isin(coded_insurance)]
df2 = df2[~df2['ever_breastfed'].isin(coded_breastfed)]

# Convert to Binary Response Variables
df2.dropna(inplace=True)

df2['underweight'] = [2 if -2 <= x < 2 else 1 for x in df2['underweight']]
df2['stunting'] = [2 if -2 <= x < 2 else 1 for x in df2['stunting']]
df2['wasting'] = [2 if -2 <= x < 2 else 1 for x in df2['wasting']]

df2['malnurished'] = df2[['underweight', 'stunting', 'wasting']].apply(lambda x: 1 if any(i == 1 for i in x) else 0, axis=1)
df2 = df2.drop(['underweight', 'stunting', 'wasting'], axis=1)
df2.head(10)

In [ ]:
df2.info()

# Artificial Neural Network

In [ ]:
X = df2.drop(columns=['malnurished'])
y = df2['malnurished']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state=42)

In [ ]:
model = Sequential()
model.add(Dense(32, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dropout(0.2))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['recall'])

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,  
    epochs=100,            
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# from kerastuner.tuners import RandomSearch
# from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.layers import Dense, Dropout
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.losses import BinaryCrossentropy
# from tensorflow.keras.metrics import Recall
# from kerastuner.tuners import RandomSearch
# from tensorflow.keras.callbacks import EarlyStopping

# def build_model(hp):
#     model = Sequential()
#     model.add(Dense(
#         units=hp.Int('units_input', min_value=16, max_value=128, step=16),
#         activation='relu',
#         input_shape=(X_train.shape[1],)
#     ))
#     model.add(Dropout(hp.Float('dropout', 0.1, 0.5, step=0.1)))
#     model.add(Dense(
#         units=hp.Int('units_hidden', min_value=8, max_value=64, step=8),
#         activation='relu'
#     ))
#     model.add(Dense(1, activation='sigmoid'))

#     model.compile(
#         optimizer=Adam(learning_rate=hp.Choice('lr', [1e-2, 1e-3, 1e-4])),
#         loss=BinaryCrossentropy(),
#         metrics=[Recall()]
#     )
#     return model

# early_stop = EarlyStopping(
#     monitor='val_loss',
#     patience=10,
#     restore_best_weights=True
# )


# tuner = RandomSearch(
#     build_model,
#     objective='val_recall',
#     max_trials=10,
#     executions_per_trial=1,
#     directory='tuner_dir',
#     project_name='binary_class_tuning'
# )

# tuner.search(X_train, y_train, epochs=50, validation_split=0.2, callbacks=[early_stop])
# best_model = tuner.get_best_models(num_models=1)[0]

In [ ]:
y_pred_probs = best_model.predict(X_test).ravel()
y_pred = (y_pred_probs > 0.5).astype(int)

print(classification_report(y_test, y_pred))


In [ ]:
# Get predicted probabilities
y_probs = model.predict(X_test)

# Convert to class labels
y_pred = (y_probs > 0.5).astype("int")

# Print classification report
print(classification_report(y_test, y_pred))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Not Underweight', 'Underweight'], yticklabels=['Not Underweight', 'Underweight'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# Use model.predict() and flatten to 1D
y_probas = model.predict(X_test).ravel()

# Compute Average Precision Score
print(f'Average Precision: {average_precision_score(y_test, y_probas)}')

In [ ]:
# Get predicted probabilities
y_probas = model.predict(X_test).ravel()  # or model.predict(X_test)[:, 1] for softmax

# Plot using from_predictions
display = PrecisionRecallDisplay.from_predictions(
    y_test,
    y_probas,
    name="Keras Model"
)
display.plot()
plt.axhline(y=sum(y_test) / len(y_test), color='gray', linestyle='--', label='Chance')
plt.title('2-class Precision Recall Curve')
plt.legend()
plt.show()
